# Validation
Checks the run end to end: all deliveries reached bronze, replays stayed there as duplicates, silver holds each event once and a replayed event is unchanged.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("expected_bronze", "0")
dbutils.widgets.text("expected_silver", "0")

from pyspark.sql import functions as F

target = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}"
expected_bronze = int(dbutils.widgets.get("expected_bronze"))
expected_silver = int(dbutils.widgets.get("expected_silver"))

bronze = spark.table(f"{target}.events_bronze")
silver = spark.table(f"{target}.events_silver")

In [ ]:
bronze_rows = bronze.count()
silver_rows = silver.count()
unique_ids = bronze.select("event_id").distinct().count()
silver_ids = silver.select("event_id").distinct().count()
versions = bronze.distinct().groupBy("event_id").count().filter("count > 1").count()
missing = bronze.select("event_id").distinct().join(silver, "event_id", "left_anti").count()

checks = {
    "bronze has every delivery": bronze_rows == expected_bronze,
    "silver has every event": silver_rows == expected_silver,
    "bronze duplicates are the replay": bronze_rows - unique_ids == expected_bronze - expected_silver,
    "silver has no duplicates": silver_rows == silver_ids,
    "nothing missing in silver": missing == 0,
    "replayed events are identical": versions == 0,
}
for name, passed in checks.items():
    print("PASS" if passed else "FAIL", name)
print(f"bronze {bronze_rows}, unique {unique_ids}, silver {silver_rows}")
assert all(checks.values()), "validation failed"

In [ ]:
display(spark.table(f"{target}.producer_metrics").orderBy("finished_at"))
display(silver.groupBy("event_type").agg(F.count("*").alias("events"), F.round(F.sum("amount"), 2).alias("amount")).orderBy("event_type"))